# Error Analysis: BM25 vs Semantic Retrieval

Pulls the queries where each method most outperforms the other (by per-query NDCG@10), shows the query, top-3 results from each method, and the qrels-relevant docs, so we can annotate *why* one wins for each example.

Run `python -m src.eval.run_eval --run results/runs/bm25.run` and the same for `semantic.run` first (or just re-run the evaluate_run calls below directly) so the per-query CSVs exist.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.eval.metrics import evaluate_run
from src.eval.trec_format import load_qrels, load_run, load_topics

qrels = load_qrels('../data/qrels/qrels.txt')
topics = load_topics('../data/qrels/topics.tsv')
subset = pd.read_parquet('../data/raw/patents_subset.parquet').set_index('patent_id', drop=False)

bm25_run = load_run('../results/runs/bm25.run')
semantic_run = load_run('../results/runs/semantic.run')

bm25_per_query = evaluate_run(qrels, bm25_run)
semantic_per_query = evaluate_run(qrels, semantic_run)

shared_qids = sorted(set(bm25_per_query) & set(semantic_per_query))
delta_df = pd.DataFrame([
    {'query_id': q,
     'bm25_ndcg10': bm25_per_query[q]['ndcg_cut_10'],
     'semantic_ndcg10': semantic_per_query[q]['ndcg_cut_10'],
     'delta_bm25_minus_semantic': bm25_per_query[q]['ndcg_cut_10'] - semantic_per_query[q]['ndcg_cut_10']}
    for q in shared_qids
]).set_index('query_id')
delta_df.sort_values('delta_bm25_minus_semantic', ascending=False).head(10)

,bm25_ndcg10,semantic_ndcg10,delta_bm25_minus_semantic
query_id,,,
9799886,1.000000,0.000000,1.000000
9171248,0.613147,0.000000,0.613147
9716274,0.613147,0.000000,0.613147
9993874,0.543771,0.000000,0.543771
10949765,0.469279,0.000000,0.469279
10322646,0.617320,0.156426,0.460893
10316047,0.397809,0.000000,0.397809
9853318,1.000000,0.613147,0.386853
9552550,0.386853,0.000000,0.386853


In [2]:
def show_example(qid, n=3):
    print(f'Query {qid}: {topics[qid]}\n')
    relevant = set(qrels.get(qid, {}))
    print('--- BM25 top-3 ---')
    for doc_id, score in bm25_run.get(qid, [])[:n]:
        title = subset.loc[doc_id, 'title'] if doc_id in subset.index else '?'
        mark = 'RELEVANT' if doc_id in relevant else ''
        print(f'  {doc_id} ({score:.3f}) {mark}: {title}')
    print('--- Semantic top-3 ---')
    for doc_id, score in semantic_run.get(qid, [])[:n]:
        title = subset.loc[doc_id, 'title'] if doc_id in subset.index else '?'
        mark = 'RELEVANT' if doc_id in relevant else ''
        print(f'  {doc_id} ({score:.3f}) {mark}: {title}')
    print()

print('=== Top 5 queries where BM25 >> Semantic ===\n')
for qid in ['9799886', '9171248', '9716274', '9993874', '10949765']:
    show_example(qid)

=== Top 5 queries where BM25 >> Semantic ===

Query 9799886: CATHODE WITH SILVER MATERIAL AND SILICATE DOPANT AND METHOD OF PRODUCING. The present invention provides novel cathodes having a reduced resistivity and other improved electrical properties. Furthermore, this invention also presents methods of manufacturing novel electrochemical cells and novel cathodes. These novel cathodes comprise a silver material that is doped with a silicate material.

--- BM25 top-3 ---
  9184444 (194.034) RELEVANT: NOVEL ELECTRODES AND RECHARGEABLE BATTERIES
  9184440 (168.257) RELEVANT: NOVEL ELECTRODES AND RECHARGEABLE BATTERIES
  10236534 (143.234) : ELECTRICAL ENERGY STORAGE ELEMENT, METHOD AND APPARATUS FOR PRODUCING SAID ELECTRICAL ENERGY STORAGE ELEMENT
--- Semantic top-3 ---
  10326135 (0.762) : DOPED CONVERSION MATERIALS FOR SECONDARY BATTERY CATHODES
  9812710 (0.759) : COPPER DOPED SVO COMPOUNDS AS HIGH RATE CATHODE MATERIALS
  10439206 (0.752) : METHOD AND MATERIAL FOR LITHIUM ION BATTERY 

In [3]:
print('=== Top 5 queries where Semantic >> BM25 ===\n')
for qid in ['10128511', '9911518', '10556494', '10892492', '10586966']:
    show_example(qid)

=== Top 5 queries where Semantic >> BM25 ===

Query 10128511: CATHODE FOR METAL-AIR BATTERY, METHOD OF MANUFACTURING THE SAME, AND METAL-AIR BATTERY COMPRISING THE SAME. The present invention relates to a cathode for a metal-air battery, a method for manufacturing the same, and a metal-air battery including the same. The cathode comprises a needle-shaped core including two or more species of metals selected from the group consisting of nickel, cobalt, manganese, zinc, iron, copper, and chrome, or a cobalt oxide; and a flake-shaped shell including an oxide containing two or more species of metals selected from the group consisting of nickel, cobalt, manganese, zinc, iron, copper, and chrome or a cobalt oxide. As such, the core-shell structure may lead to a reduction in the charge voltage of the metal-air battery as well as the taking of the good capacity characteristics of the transition metal oxide. Further, according to the present invention, the cathode for a metal-air battery may be

## Failure Modes

**BM25 wins when the relevant document shares specific, somewhat rare technical
terminology with the query that gives BM25 a strong exact-match signal, while
semantic retrieval clusters within the broad topic and surfaces other
plausible-but-uncited patents instead.** Across all 5 BM25-wins examples,
semantic's top-3 results are genuinely on-topic (neuromorphic circuits for a
neuromorphic-circuit query, sodium-battery cathodes for a sodium-battery
query) but consistently miss the *specific* cited document -- the embedding
model captures "this is about neuromorphic computing" well but doesn't
discriminate the finer-grained technical connection (e.g. "digital-to-analogue
converter" as the specific shared mechanism) that actually drove the citation.
BM25's exact IDF-weighted term matching catches that specific overlap
directly.

**Semantic wins for three distinct, identifiable reasons:**

1. **Vocabulary mismatch across specific-instance/general-category pairs.**
   Query 10128511 says "METAL-AIR BATTERY"; the relevant doc says
   "LITHIUM-AIR BATTERY" (lithium being a specific metal). BM25's exact terms
   don't bridge "metal" vs "lithium"; the embedding correctly treats them as
   closely related.

2. **BM25's IDF weighting can backfire in a domain-narrow corpus.** Query
   9911518's title is "Cathode Active Material For Lithium-Ion Battery"; the
   relevant doc's title is *almost word-for-word identical*
   ("Cathode Active Material For Lithium Ion Battery, Cathode For Lithium Ion
   Battery, And Lithium Battery") -- yet BM25 doesn't rank it in its top-3.
   Because "cathode," "lithium," "ion," and "battery" are ubiquitous across
   this H01M-heavy corpus, IDF down-weights exactly the terms that make this
   pair an obvious match. Semantic embeddings aren't penalized this way and
   catch it at rank 1 (score 0.964).

3. **A genuine, fixable tokenization bug: hyphenation.** Query 10556494 is
   titled "AIR-COOLED FUEL CELL VEHICLE"; the relevant doc is
   "AIR-COOLED FUEL-CELL VEHICLE" -- a near-exact title match differing only
   by a hyphen. Checked directly:
   `tokenize('...FUEL CELL...')` → `['fuel', 'cell']` but
   `tokenize('...FUEL-CELL...')` → `['fuel-cell']` (one token, since
   `src/index/bm25_index.py`'s regex keeps internal hyphens attached to the
   token). These never overlap under BM25's bag-of-words scoring, so a
   virtually identical title gets zero credit. Semantic embedding doesn't
   tokenize this way and catches it at rank 1. **Actionable fix, not
   implemented here:** either strip hyphens before tokenizing, or emit both
   the hyphenated and de-hyphenated forms as tokens, so "fuel-cell" and
   "fuel cell" match. Flagged as a concrete, scoped future-work item rather
   than a vague "BM25 has limitations" statement.

4. (Query 10892492) Semantic also wins on more subtle conceptual
   relationships that share little surface vocabulary at all -- the query's
   abstract emphasizes an "essentially carbon-free cathode," while the
   relevant doc is about "COATING OF POROUS CARBON" for the same battery
   subfield. These read as almost contradictory at the lexical level
   (carbon-free vs. porous carbon) but are topically close (both concern
   carbon's role in lithium-air cathode design) -- exactly the kind of
   relationship embeddings are supposed to catch and literal term matching
   structurally cannot.
